# Multi-competition data processing
Domestic leagues (football-data.co.uk) and Champions League (fixturedownload.com), from the raw files of `data/raw/` to `data/processed/`.

Raw files are downloaded by `src/download.py`.

## 1. Raw Files
One folder per competition, one file per season.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

#Reuse the competitions config of the project
sys.path.append("../src")
from competitions import LEAGUES, CUPS, UCL_ALIASES

RAW_DIR = Path("../data/raw/")
PROCESSED_DIR = Path("../data/processed/")

for league in LEAGUES.values():
    print(league["folder"], len(list((RAW_DIR / league["folder"]).glob("season-*.csv"))), "seasons")
for cup in CUPS.values():
    print(cup["folder"], len(list((RAW_DIR / cup["folder"]).glob("season-*.json"))), "seasons")

## 2. Domestic Leagues: Consolidation
Same metrics as Ligue 1, plus a `Competition` column (football-data division code).

In [ ]:
COLS_TO_KEEP = [
    "Competition", "Season", "Date", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR",
    "HS", "AS", "HST", "AST",
    "HC", "AC", "HY", "AY", "HR", "AR"
]

dfs = []

#Adding COMPETITION and SEASON columns, then concat every league and season
for div, league in LEAGUES.items():
    for file_path in sorted((RAW_DIR / league["folder"]).glob("season-*.csv")):
        df_season = pd.read_csv(file_path, encoding="utf-8-sig", usecols=COLS_TO_KEEP[2:])
        df_season["Competition"] = div
        df_season["Season"] = file_path.stem[7:11]
        #Keeping only the good columns, and the rows of played matches
        df_season = df_season[COLS_TO_KEEP].dropna(subset=["HomeTeam", "FTR"])
        dfs.append(df_season)

df_leagues = pd.concat(dfs, ignore_index=True)
print(df_leagues.shape)
print(df_leagues.groupby("Competition").size())
print(df_leagues.isna().sum()) #-> A few matches have no stats: kept, stored as NULL

## 3. Domestic Leagues: Cleaning & Schema Normalization
Two date formats in the raw files: `2025-08-15` (Ligue 1 history) and `15/08/2025` (football-data).

Each format is parsed explicitly: with `dayfirst=True` pandas reads `2019-08-09` as September 8th.

In [ ]:
#Parse each format on its own rows
iso = df_leagues["Date"].str.match(r"\d{4}-")
df_leagues["Date"] = pd.concat([
    pd.to_datetime(df_leagues.loc[iso, "Date"], format="%Y-%m-%d"),
    pd.to_datetime(df_leagues.loc[~iso, "Date"], format="%d/%m/%Y"),
]).sort_index()

#Sanity check: every match is inside its season (July -> June)
start_year = 2000 + df_leagues["Season"].str[:2].astype(int)
in_season = (df_leagues["Date"] >= pd.to_datetime(start_year.astype(str) + "-07-01")) & \
            (df_leagues["Date"] < pd.to_datetime((start_year + 1).astype(str) + "-09-01"))
print("Matches outside their season:", (~in_season).sum())

rename_mapping = {
    "Competition": "competition",
    "Season": "season",
    "Date": "match_date",
    "HomeTeam": "home_team",
    "AwayTeam": "away_team",
    "FTHG": "home_goals",
    "FTAG": "away_goals",
    "FTR": "result",
    "HS": "home_shots",
    "AS": "away_shots",
    "HST": "home_shots_on_target",
    "AST": "away_shots_on_target",
    "HC": "home_corners",
    "AC": "away_corners",
    "HY": "home_yellow_cards",
    "AY": "away_yellow_cards",
    "HR": "home_red_cards",
    "AR": "away_red_cards",
}
df_leagues = df_leagues.rename(columns=rename_mapping)

#Strip invisible whitespace from team names
df_leagues["home_team"] = df_leagues["home_team"].str.strip()
df_leagues["away_team"] = df_leagues["away_team"].str.strip()

#Goals are always there, stats can be missing: nullable integers
df_leagues[["home_goals", "away_goals"]] = df_leagues[["home_goals", "away_goals"]].astype(int)
stat_cols = list(rename_mapping.values())[8:]
df_leagues[stat_cols] = df_leagues[stat_cols].astype("Int64")

df_leagues = df_leagues.sort_values(["match_date", "competition"]).reset_index(drop=True)
display(df_leagues.head(10))

## 4. Champions League: Consolidation
Scores only (no shots). Club names are mapped to the football-data names so that a club has the same name in its league and in Europe.

Neutral venue: every final, and the 2020 Final 8 played in Lisbon.

In [ ]:
rows = []

for file_path in sorted((RAW_DIR / "champions-league").glob("season-*.json")):
    season = file_path.stem[7:11]
    matches = json.loads(file_path.read_text())

    #The final is the last round of a finished season
    finished = all(m["HomeTeamScore"] is not None for m in matches)
    last_round = max(m["RoundNumber"] for m in matches)

    for m in matches:
        #Upcoming matches are not history
        if m["HomeTeamScore"] is None:
            continue
        date = pd.to_datetime(m["DateUtc"]).tz_localize(None)
        hg, ag = m["HomeTeamScore"], m["AwayTeamScore"]
        rows.append({
            "competition": "UCL",
            "season": season,
            "match_date": date.normalize(),
            "round": m["RoundNumber"],
            "neutral": (finished and m["RoundNumber"] == last_round) or (season == "1920" and date >= pd.Timestamp("2020-08-12")),
            "home_team": UCL_ALIASES.get(m["HomeTeam"], m["HomeTeam"]),
            "away_team": UCL_ALIASES.get(m["AwayTeam"], m["AwayTeam"]),
            "home_goals": hg,
            "away_goals": ag,
            "result": "H" if hg > ag else "A" if hg < ag else "D",
        })

df_ucl = pd.DataFrame(rows).sort_values("match_date").reset_index(drop=True)
print(df_ucl.shape)
print(df_ucl.groupby("season").size())
print("Neutral venue matches:", df_ucl["neutral"].sum())
display(df_ucl.tail(10))

## 5. Champions League: Coverage
Which clubs have domestic history? The others only have their European matches: the features will fall back on default values for them.

In [ ]:
league_teams = set(df_leagues["home_team"]) | set(df_leagues["away_team"])
ucl_teams = set(df_ucl["home_team"]) | set(df_ucl["away_team"])

covered = sorted(ucl_teams & league_teams)
uncovered = sorted(ucl_teams - league_teams)
print(f"{len(covered)} clubs with domestic history, {len(uncovered)} without")
print(uncovered)

#Share of Champions League matches where both clubs have domestic history
both = df_ucl["home_team"].isin(league_teams) & df_ucl["away_team"].isin(league_teams)
print(f"Both clubs covered in {both.mean():.0%} of the matches")

## 6. Save

In [ ]:
df_leagues.to_csv(PROCESSED_DIR / "league_matches.csv", index=False)
df_ucl.to_csv(PROCESSED_DIR / "ucl_matches.csv", index=False)
print("Leagues and Champions League saved!")